In [25]:

from imitation.scripts.generate_ranked_trajectories import generate_policies_and_videos, generate_trajectories, record_trajectory_video 

ENV_NAME = "CartPole-v1"
ITERATION_STEP = 1000
NUM_POLICIES = 10

video_entries, policy_entries = generate_policies_and_videos(NUM_POLICIES, ITERATION_STEP, ENV_NAME)
trajectory_entries = generate_trajectories(policy_entries, ENV_NAME)

for i in range(0, len(trajectory_entries)):
    print(len(trajectory_entries[i]["trajectory"].obs)) # why is reward capped at 500 ? 


ppo_cartpole_1000 already exists.
ppo_cartpole_2000 already exists.
ppo_cartpole_3000 already exists.
ppo_cartpole_4000 already exists.
ppo_cartpole_5000 already exists.
ppo_cartpole_6000 already exists.
ppo_cartpole_7000 already exists.
ppo_cartpole_8000 already exists.
ppo_cartpole_9000 already exists.
ppo_cartpole_10000 already exists.
61
53
64
144
150
146
247
271
481
178
501
501
230
225
187
501
501
501
175
137
138
501
452
501
501
192
501
501
501
501


In [ ]:
# Building ranked pairs as a learning dataset
pairs = []
for i, t1 in enumerate(trajectory_entries):
    for j, t2 in enumerate(trajectory_entries):
        if i >= j: continue
        y = 1.0 if sum(t1["trajectory"].rews) > sum(t2["trajectory"].rews) else 0.0
        pairs.append((t1["trajectory"].obs, t2["trajectory"].obs, y))

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class PrefPairsDS(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        o1, o2, y = self.pairs[i]               # (T, obs_dim), (T, obs_dim), scalar
        return torch.from_numpy(o1), torch.from_numpy(o2), torch.tensor(y, dtype=torch.float32)


train_loader = DataLoader(PrefPairsDS(pairs), batch_size=64, shuffle=True)

In [33]:
# setup neural network and loss function
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

obs_dim = gym.make(ENV_NAME).observation_space.shape[0]

class RewardMLP(nn.Module):
    def __init__(self, obs_dim, hid=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hid), nn.Tanh(),
            nn.Linear(hid, hid), nn.Tanh(),
            nn.Linear(hid, 1)
        )
    def forward(self, x):            # x: (B, D)
        return self.net(x).squeeze(-1)

reward_net = RewardMLP(obs_dim)
opt = torch.optim.Adam(reward_net.parameters(), lr=3e-4)
device = torch.device("cpu"); reward_net.to(device)

def pred_frag_return(obs_seq):       # (B, T, D) -> (B,)
    B,T,D = obs_seq.shape
    r = reward_net(obs_seq.reshape(B*T, D)).view(B,T).sum(dim=1)
    return r

def bt_loss(r1, r2, y):              # y in {0,1}
    return F.binary_cross_entropy_with_logits(r1 - r2, y)

